In [1]:
import pandas as pd
enron = pd.read_csv('../data/enron_model_ft.csv')

/var/folders/4y/0t9f134178bbfj5m299wl_7h0000gn/T/ipykernel_13162/2732611547.py:2: DtypeWarning: Columns (2,4,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  enron = pd.read_csv('../data/enron_model_ft.csv')


In [7]:
import os
from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv('../.env')

URI = os.getenv("NEO4J_URI")
AUTH = (os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))

driver = GraphDatabase.driver(URI, auth=AUTH)

# quick connection test
driver.verify_connectivity()
print("Connected!")

Connected!


In [8]:
def bucket_urgency(minutes):
    if pd.isna(minutes):
        return 'ghosted'
    elif minutes < 5:
        return 'instant'
    elif minutes < 15:
        return 'fast'
    elif minutes < 24*60:
        return 'same_day'
    else:
        return 'next_day'

enron['urgency_bucket'] = enron['response_time'].apply(bucket_urgency)

In [9]:
edge_data = enron.dropna(subset=['response_time']).groupby(['From', 'To']).agg(
    count=('response_time', 'size'),
    urgency=('urgency_bucket', lambda x: x.mode()[0])
).reset_index()

edge_data = edge_data.sort_values('count', ascending=False).head(50)
print(f"{len(edge_data)} relationships ready to load")

50 relationships ready to load


In [10]:
with driver.session() as session:
    session.run("""
        UNWIND $rows AS row
        MERGE (a:Person {email: row.From})
        MERGE (b:Person {email: row.To})
        MERGE (a)-[r:EMAILED]->(b)
        SET r.urgency = row.urgency, r.count = row.count
    """, rows=edge_data.to_dict('records'))

print("Loaded into Neo4j")

Loaded into Neo4j


In [11]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

g = Neo4jGraphWidget(driver)
g.show_cypher("MATCH (a)-[r]->(b) RETURN a, r, b")

GraphWidget(layout=Layout(height='800px', width='100%'))